# A straight line first: linear regression

Before the neural network, the simplest model that can predict `y` from `x` — on our own data. Take one input, the
eastward wind `uas`, and one output, the eastward ice velocity `siu`, along the ITP tracks of one member.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from oaisis import data, plots, ml, metrics, selectors, experiments, teaching

In [ ]:
xs, ys = teaching.xy(member="r1i1p1f1", x="uas", y="siu", n=2000)    # 2000 random ITP samples
teaching.scatter(xs, ys, title="How do you predict y from x?");

## The model: y = w x + b

A straight line has two **parameters**: the weight `w` (slope) and the bias `b` (intercept). For a given `w` and `b`
the model's prediction for sample *i* is $\hat y_i = w\,x_i + b$, and its error is $e_i = \hat y_i - y_i$.

In [ ]:
w, b = teaching.fit_line(xs, ys)          # least squares: the w, b that minimize the mean squared error
teaching.scatter(xs, ys, model=(w, b), residuals=12);

The slope says ice moves at about 2 % of the wind speed — the classic *Nansen rule*, found by Fridtjof Nansen from
the drift of the *Fram* in the 1890s. The intercept is slightly negative: on average the ice drifts a little westward
even with no eastward wind (the Beaufort Gyre, and the ocean under it).

## The loss function

"Best line" has to mean something precise. We measure how wrong the model is with a **loss function**, here the
**mean squared error** over all $N$ training samples:

$$ J(w, b) = \frac{1}{N}\sum_{i=1}^{N} \left(w\,x_i + b - y_i\right)^2 $$

Training *is* minimizing $J$: the black arrows in the plot above are the $e_i$, and the best line is the one that makes
their squares, summed, as small as possible. For a straight line this has a closed-form answer (that is what `fit_line`
used). Picture $J$ as a landscape over the parameters — here a bowl over $(w, b)$ with a single lowest point — and
training as finding that point. A neural network has thousands of parameters instead of two, its landscape is bumpy, and
there is no closed form; but training it means exactly the same thing, done step by step. That is the next page.

## All seven inputs

Linear regression works just as well with several inputs: $\mathbf{Y} = \mathbf{W}\mathbf{X} + \mathbf{B}$ with a weight
per input and per output. Fitted on the training members and scored on the test members, it is our **baseline**: any
network that cannot beat this is not worth its complexity.

In [ ]:
from sklearn.linear_model import LinearRegression

p = data.prepared()                                     # the split and scaled tables from the previous page
linear = LinearRegression().fit(p.x_tr, p.y_tr)
Y_lin = p.scaler.y_inv(linear.predict(p.x_te))          # back to m/s
metrics.report(p.Y_te, Y_lin, label="linear");

The **skill** reported here is $1 - \mathrm{RMS}(\text{vector error}) / \mathrm{RMS}(\text{anomaly of the truth})$:
1 is perfect, 0 means the model is no better than predicting the mean velocity everywhere, negative is worse than that.
We score the *vector* (u, v) rather than the components separately, because that is what matters for ice drift.
A skill of about 0.66 with a correlation above 0.9 is already good — the problem is largely linear.

The weights themselves are interpretable — they act on the scaled inputs:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(np.arange(7) - 0.2, linear.coef_[0], 0.4, label="-> siu (eastward ice)")
ax.bar(np.arange(7) + 0.2, linear.coef_[1], 0.4, label="-> siv (northward ice)")
ax.set_xticks(range(7)); ax.set_xticklabels(data.FEATURES); ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("weight W (scaled units)"); ax.legend(frameon=False);

Eastward wind drives eastward ice and northward wind drives northward ice, with a smaller cross term: the ice moves
slightly to the *right* of the wind, the Coriolis effect in the momentum balance. The large weight on latitude is less
dramatic than it looks: scaled latitude only varies between 0.79 and 1.0, so this weight mostly works together with the
intercept to set the mean westward drift of the Beaufort Gyre. Everything else is a small correction — in a *linear* model. Whether non-linear combinations of the inputs carry more information is what the
network will tell us.

Next: [the neural network](04_neural_network.ipynb).